# MultiTask Face Analysis Model Evaluation Notebook

In [ ]:
import numpy as np, eval, matplotlib.pyplot as plt, os
import torch, torchvision, datasets
import torch.nn as nn
import seaborn as sns, cv2
from torchvision.transforms import v2
from inference_DaVinciFace import DaVinciFace
from configs.DaVinciFace.train_davit_t_face_age_gender_race_emotion_attribute_pose_landmarks_visibility_DaVinciFace_channel_cross_attention_glint360k_pretrained_4_layer_decoder import config
from utility_scripts.model_stats import get_model_stats_complete
from pytorch_grad_cam.utils.image import show_cam_on_image, preprocess_image


/home/thoal/anaconda3/envs/pytorch/lib/python3.13/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
backbone_name = config['backbone_name']
decoder_type = config['decoder_type']
num_decoder_layers = config['num_decoder_layers']
output_folder_name = config['output_folder_name']

In [3]:
backbone_name, decoder_type, num_decoder_layers

('davit_t', 'channel_cross_attention', 4)

In [4]:
print(config)

{'output_folder_name': 'train_davit_t_face_age_gender_race_emotion_attribute_pose_landmarks_visibility_DaVinciFace_channel_cross_attention_glint360k_pretrained_4_layer_decoder', 'backbone_name': 'davit_t', 'compile_model': True, 'pretrained_backbone_path': 'data/models/pretrained models/davit_t_adaface_Glint360k/backbone.pth', 'pretrained_face_recognition_subnet_path': 'data/models/pretrained models/davit_t_adaface_Glint360k/recognition_subnet.pth', 'pretrained_margin_head_path': 'data/models/pretrained models/davit_t_adaface_Glint360k/margin_head.pth', 'head_type': 'adaface', 'embedding_dim': 512, 'transformer_embedding_dim': 256, 'num_decoder_layers': 4, 'num_heads': 8, 'mlp_ratio': 8.0, 'decoder_type': 'channel_cross_attention', 'margin': 0.4, 'scale': 64, 'h': 0.333, 't_alpha': 0.05, 'optimizer': 'adamw', 'learning_rate': 0.0005, 'backbone_lr_multiplier': 0.1, 'weight_decay': 0.05, 'gradient_clip_val': 1.0, 'scheduler': 'cosine', 'cosine_restart_start': 15, 'cosine_restart_mult': 1

In [5]:
model = DaVinciFace(**config)

Loaded pretrained backbone from data/models/pretrained models/davit_t_adaface_Glint360k/backbone.pth.
Loaded pretrained face recognition subnet from data/models/pretrained models/davit_t_adaface_Glint360k/recognition_subnet.pth
Initialized DaVinciFace with the following configuration:
  output_folder_name: train_davit_t_face_age_gender_race_emotion_attribute_pose_landmarks_visibility_DaVinciFace_channel_cross_attention_glint360k_pretrained_4_layer_decoder
  backbone_name: davit_t
  compile_model: True
  pretrained_backbone_path: data/models/pretrained models/davit_t_adaface_Glint360k/backbone.pth
  pretrained_face_recognition_subnet_path: data/models/pretrained models/davit_t_adaface_Glint360k/recognition_subnet.pth
  pretrained_margin_head_path: data/models/pretrained models/davit_t_adaface_Glint360k/margin_head.pth
  head_type: adaface
  embedding_dim: 512
  transformer_embedding_dim: 256
  num_decoder_layers: 4
  num_heads: 8
  mlp_ratio: 8.0
  decoder_type: channel_cross_attention


In [6]:
state_dict = torch.load(os.path.join('data', 'models', output_folder_name, 'model.pth'))
state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()} # remove '_orig_mod.' prefix if it exists

new_state_dict = {}
for k, v in state_dict.items():
    if 'margin' not in k: # exclude margin head weights
        new_state_dict[k] = v
    
model.load_state_dict( # load the complete model
    new_state_dict
)

<All keys matched successfully>

## Get model stats for the inference model

In [7]:
image_size = (112, 112)

In [ ]:
params_count, flops_count, latency, fps, peak_mem_batch_mb, peak_mem_single_mb = get_model_stats_complete(model, num_warmup = 250, num_measure = 500, input_res = (3, *image_size))

In [9]:
print(f'Params: {params_count}')
print(f'FLOPs: {flops_count}')
print(f'latency: {latency}')
print(f'FPS: {fps}')
print(f'Peak Memory (Batch): {peak_mem_batch_mb} MB')
print(f'Peak Memory (Single): {peak_mem_single_mb} MB')

Params: 42.340068
FLOPs: 8.469557504
latency: 6.759624381595188
FPS: 2984.371347138715
Peak Memory (Batch): 2755.994140625 MB
Peak Memory (Single): 1069.71728515625 MB


### Save the model without the margin head for inference

In [10]:
# remove the margin head
model_dict = model.state_dict()
new_dict = {}
for key in model_dict:
    if 'margin' not in key:
        new_dict[key] = model_dict[key]

In [11]:
ready_to_share_weights_dir = os.path.join('data', 'models', 'ready to share weights', output_folder_name)
if not os.path.exists(ready_to_share_weights_dir):
    os.makedirs(ready_to_share_weights_dir, exist_ok = True)
    
torch.save(
    new_dict,
    os.path.join(ready_to_share_weights_dir, 'model.pth')
)

In [9]:
results_dir = os.path.join('data', 'results', output_folder_name)
if not os.path.exists(results_dir):
    os.makedirs(results_dir, exist_ok = True)

In [10]:
test_transform = v2.Compose([ # for testing on datasets other than face recognition.
    v2.ToImage(),
    v2.Resize(image_size),
    v2.ToDtype(torch.float32, scale = True),
    v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

In [14]:
if not os.path.exists(os.path.join(results_dir, 'model_complexity.txt')):
    open(os.path.join(results_dir, 'model_complexity.txt'), 'x')

In [15]:
with open(os.path.join(results_dir, 'model_complexity.txt'), 'w') as f:
    f.write(f"Total FLOPs: {flops_count} GFLOPs\n")
    f.write(f"Total Parameters: {params_count} M\n")
    f.write(f'Latency: {latency}ms \n')
    f.write(f'Throughput: {fps} fps\n')
    f.write(f'Peak Memory (Batch): {peak_mem_batch_mb} MB\n')
    f.write(f'Peak Memory (Single): {peak_mem_single_mb} MB\n')

## Face Recognition

In [16]:
face_recognition_model = nn.Sequential(
    model.backbone,
    model.face_recognition_embedding_subnet
)

In [17]:
metrics = eval.evaluate_face_recognition(face_recognition_model, image_size = image_size[0])

Processing dataset: LFW...
Finished processing LFW.
Processing dataset: CPLFW...
Finished processing CPLFW.
Processing dataset: CALFW...
Finished processing CALFW.
Processing dataset: CFP-FP...
Finished processing CFP-FP.
Processing dataset: CFP-FF...
Finished processing CFP-FF.
Processing dataset: AgeDB30...
Finished processing AgeDB30.
Processing dataset: VGG2FP...
Finished processing VGG2FP.


In [18]:
for key, db_metrics in metrics.items():
    accuracy, precision, recall, f1_score, auc_score, fpr, tpr, roc_thresholds = db_metrics
    
    face_rec_results_output_dir = os.path.join(results_dir, 'face verification', f'{key}')
    os.makedirs(face_rec_results_output_dir, exist_ok = True)

    print(f'Accuracy for {key} = {accuracy}.')
    with open(os.path.join(face_rec_results_output_dir, 'results.txt'), 'w') as file:
        result_txt_output = f'Accuracy for {key} = {accuracy}\n'
        result_txt_output += f'F1 score for {key} = {f1_score}\n'
        result_txt_output += f'AUC score for {key} = {auc_score}\n'
        result_txt_output += f'Precision for {key} = {precision}\n'
        result_txt_output += f'Recall for {key} = {recall}\n'
        file.write(result_txt_output)
    
    # 1. Calculate the Equal Error Rate (EER) Point
    # This is where False Accept Rate (fpr) ~ False Reject Rate (1 - tpr)
    # This corresponds to the intersection with the diagonal y = 1 - x
    fnr = 1 - tpr
    eer_index = np.nanargmin(np.absolute((fnr - fpr)))
    eer_fpr = fpr[eer_index]
    eer_tpr = tpr[eer_index]
    
    plt.figure(figsize=(8, 8), dpi = 800)
    plt.title(f'ROC Curve for {key}', fontsize=14, fontweight='bold')
    
    # 2. Plot the Random Model (Gray Dashed Line)
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', alpha=0.7, label='Random Guessing')

    # 3. Plot the anti-random line
    plt.plot([0, 1], [1, 0], linestyle = '-.', color = 'gray', alpha = 0.7)

    # 4. Plot the Main Model (Royal Blue)
    # Using 'zorder' to ensure it sits on top of the grid/random line
    plt.plot(fpr, tpr, color='#4169E1', linewidth=2, label=f'AUC: {auc_score:.4f}', zorder=3)
    
    # 5. Plot the Red Dot at EER
    plt.scatter(eer_fpr, eer_tpr, color='red', s=50, label=f'EER Point (FPR={eer_fpr:.2f})', zorder=4)
    
    # Styling
    plt.legend(loc='lower right', fontsize=10)
    plt.xlabel('False Positive Rate (FPR)', fontsize=12)
    plt.ylabel('True Positive Rate (TPR)', fontsize=12)
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.xlim([-0.02, 1.02])
    plt.ylim([-0.02, 1.02])
    
    plt.tight_layout()
    plt.savefig(
        os.path.join(face_rec_results_output_dir, f'ROC Curve for {key}.png')
    )
    plt.close()

Accuracy for LFW = 0.9983333333333168.
Accuracy for CPLFW = 0.9319999999999846.
Accuracy for CALFW = 0.958999999999984.
Accuracy for CFP-FP = 0.9669999999999863.
Accuracy for CFP-FF = 0.9982857142857.
Accuracy for AgeDB30 = 0.9774999999999837.
Accuracy for VGG2FP = 0.9461999999999812.


## Emotion Recognition

In [19]:
emotion_result_output_dir = os.path.join(results_dir, 'Emotion Recognition')
os.makedirs(emotion_result_output_dir, exist_ok = True)

In [20]:
rafdb_test = datasets.RAFDB(subset = 'test', transform = test_transform)

In [21]:
raf_test_dataloader = torch.utils.data.DataLoader(
    dataset = rafdb_test, 
    batch_size = 32,
    num_workers = 2,
)

In [22]:
emotion_recognition_accuracy, emotion_recognition_f1, emotion_recognition_precision, emotion_recognition_recall, cm = eval.evaluate_emotion(model = model, dataloader = raf_test_dataloader)

In [23]:
with open(os.path.join(emotion_result_output_dir, 'results.txt'), 'w') as file:
    result_string = f'Accuracy for RAFDB: {emotion_recognition_accuracy}\n'
    result_string += f'F1 Score for RAFDB: {emotion_recognition_f1}\n'
    result_string += f'Precision for RAFDB: {emotion_recognition_precision}\n'
    result_string += f'Recall for RAFDB: {emotion_recognition_recall}'
    file.write(result_string)

print(result_string)

Accuracy for RAFDB: 0.9087353324641461
F1 Score for RAFDB: 0.907913348717681
Precision for RAFDB: 0.9078808233324384
Recall for RAFDB: 0.9087353324641461


In [24]:
class_names = ['neutral', 'happy', 'sad', 'surprise', 'fear', 'disgust', 'anger']

In [25]:
plt.figure(figsize=(10, 8), dpi=800)
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.ylabel('True Label', fontsize=12, labelpad = 30)
plt.xlabel('Predicted Label', fontsize=12, labelpad = 30)
plt.title('Confusion Matrix for RAF-DB (Emotion Recognition)', fontsize=14, pad = 30)
plt.savefig(
    os.path.join(emotion_result_output_dir, 'Confusion Matrix RAFDB.png')
)
plt.close()

# Age Estimation

In [26]:
age_results_output_dir = os.path.join(results_dir, 'Age Estimation')
os.makedirs(age_results_output_dir, exist_ok=True)

### UTKFace

In [27]:
utk_face_results_dir = os.path.join(age_results_output_dir, 'UTKFace')
os.makedirs(utk_face_results_dir, exist_ok = True)

In [28]:
utk_face_db = datasets.UTKFace(subset = 'test', transform = test_transform)
utk_face_dataloader = torch.utils.data.DataLoader(
    dataset = utk_face_db,
    batch_size = 32,
    num_workers = 2
)

In [29]:
age_mae, true_ages, predicted_ages_list = eval.evaluate_age(model, utk_face_dataloader)

In [30]:
with open(os.path.join(utk_face_results_dir, 'results.txt'), 'w') as file:
    file.write(f'MAE for UTKFace: {age_mae}')
print(f'MAE for UTKFace: {age_mae}')

MAE for UTKFace: 4.084355354309082


In [31]:
plt.figure(figsize=(10, 10), dpi=600)

# Scatter plot with transparency to show density
plt.scatter(true_ages, predicted_ages_list, alpha=0.2, s=5, c='blue', label='Predictions')

# Ideal line
plt.plot([0, 100], [0, 100], 'r--', linewidth=2, label='Perfect Prediction')

plt.xlabel('Ground Truth Age (Years)')
plt.ylabel('Predicted Age (Years)')
plt.title(f'Age Estimation Scatter Plot for UTKFace (MAE: {age_mae:.4f})', fontsize=14)
plt.legend(loc = 'lower right')
plt.grid(True, linestyle=':', alpha=0.6)

plt.savefig(
    os.path.join(utk_face_results_dir, 'Age Estimation Scatter Plot for UTKFace')
)
plt.close()

### MORPH

In [32]:
morph_results_dir = os.path.join(age_results_output_dir, 'MORPH')
os.makedirs(morph_results_dir, exist_ok = True)

In [33]:
morph_db = datasets.MORPH(subset = 'test', transform = test_transform)
morph_dataloader = torch.utils.data.DataLoader(
    dataset = morph_db,
    batch_size = 32,
    num_workers = 2
)

In [34]:
age_mae, true_ages, predicted_ages_list = eval.evaluate_age(model, morph_dataloader)

In [35]:
with open(os.path.join(morph_results_dir, 'results.txt'), 'w') as file:
    file.write(f'MAE for MORPH: {age_mae}')
print(f'MAE for MORPH: {age_mae}')

MAE for MORPH: 1.6925554275512695


In [36]:
plt.figure(figsize=(10, 10), dpi=600)

# Scatter plot with transparency to show density
plt.scatter(true_ages, predicted_ages_list, alpha=0.2, s=5, c='blue', label='Predictions')

# Ideal line
plt.plot([0, 100], [0, 100], 'r--', linewidth=2, label='Perfect Prediction')

plt.xlabel('Ground Truth Age (Years)')
plt.ylabel('Predicted Age (Years)')
plt.title(f'Age Estimation Scatter Plot for MORPH (MAE: {age_mae:.4f})', fontsize=14)
plt.legend(loc = 'lower right')
plt.grid(True, linestyle=':', alpha=0.6)

plt.savefig(
    os.path.join(morph_results_dir, 'Age Estimation Scatter Plot for MORPH')
)
plt.close()

### AgeDB 

In [37]:
agedb_results_dir = os.path.join(age_results_output_dir, 'AgeDB')
os.makedirs(agedb_results_dir, exist_ok = True)

In [38]:
agedb_dataset = datasets.AgeDB(transform = test_transform)
agedb_dataloader = torch.utils.data.DataLoader(
    dataset = agedb_dataset,
    batch_size = 32,
    num_workers = 2
)

In [39]:
age_mae, true_ages, predicted_ages_list = eval.evaluate_age(model, agedb_dataloader)

Validating Age:   0%|          | 0/516 [00:00<?, ?it/s]

In [40]:
np.sum(predicted_ages_list - true_ages) / len(predicted_ages_list) # check if the model overshoots the age or undershoots.

np.float32(-0.67582816)

In [41]:
with open(os.path.join(agedb_results_dir, 'results.txt'), 'w') as file:
    file.write(f'MAE for AgeDB: {age_mae}')
print(f'MAE for AgeDB: {age_mae}')

MAE for AgeDB: 5.191845893859863


In [42]:
plt.figure(figsize=(10, 10), dpi=600)

# Scatter plot with transparency to show density
plt.scatter(true_ages, predicted_ages_list, alpha=0.2, s=5, c='blue', label='Predictions')

# Ideal line
plt.plot([0, 100], [0, 100], 'r--', linewidth=2, label='Perfect Prediction')

plt.xlabel('Ground Truth Age (Years)')
plt.ylabel('Predicted Age (Years)')
plt.title(f'Age Estimation Scatter Plot for AgeDB (MAE: {age_mae:.4f})', fontsize=14)
plt.legend(loc = 'lower right')
plt.grid(True, linestyle=':', alpha=0.6)

plt.savefig(
    os.path.join(agedb_results_dir, 'Age Estimation Scatter Plot for AgeDB')
)
plt.close()

## Gender Recognition

### UTKFace

In [43]:
gender_recognition_results_dir = os.path.join(results_dir, 'Gender Recognition')
os.makedirs(gender_recognition_results_dir, exist_ok = True)

In [44]:
utk_face_results_dir = os.path.join(gender_recognition_results_dir, 'UTKFace')
os.makedirs(utk_face_results_dir, exist_ok = True)

In [45]:
gender_accuracy, gender_f1, gender_precision, gender_recall, gender_cm, fpr, tpr, auc_score, thresholds = eval.evaluate_gender(model, utk_face_dataloader)

Validating Gender:   0%|          | 0/75 [00:00<?, ?it/s]

In [46]:
with open(os.path.join(utk_face_results_dir, 'results.txt'), 'w') as file:
    result_txt_output = f'Accuracy for UTKFace (Gender Recognition): {gender_accuracy}\n'
    result_txt_output += f'F1 Score for UTKFace (Gender Recognition): {gender_f1}\n'
    result_txt_output += f'AUC Score for UTKFace (Gender Recognition): {auc_score}\n'
    result_txt_output += f'Precision for UTKFace (Gender Recognition): {gender_precision}\n'
    result_txt_output += f'Recall for UTKFace (Gender Recognition): {gender_recall}\n'
    file.write(result_txt_output)
print(result_txt_output)

Accuracy for UTKFace (Gender Recognition): 0.9611650485436893
F1 Score for UTKFace (Gender Recognition): 0.9619205298013245
AUC Score for UTKFace (Gender Recognition): 0.9853239970399901
Precision for UTKFace (Gender Recognition): 0.9579554822753503
Recall for UTKFace (Gender Recognition): 0.9659185369908562



In [47]:

# 1. Calculate the Equal Error Rate (EER) Point
# This is where False Accept Rate (fpr) ~ False Reject Rate (1 - tpr)
# This corresponds to the intersection with the diagonal y = 1 - x
fnr = 1 - tpr
eer_index = np.nanargmin(np.absolute((fnr - fpr)))
eer_fpr = fpr[eer_index]
eer_tpr = tpr[eer_index]

plt.figure(figsize=(8, 6), dpi = 800)
plt.title(f'ROC Curve for UTKFace (Gender Recognition)', fontsize=14, fontweight='bold')

# 2. Plot the Random Model (Gray Dashed Line)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', alpha=0.7, label='Random Guessing')

# 3. Plot the Main Model (Royal Blue)
# Using 'zorder' to ensure it sits on top of the grid/random line
plt.plot(fpr, tpr, color='#4169E1', linewidth=2, label=f'AUC: {auc_score:.4f}', zorder=3)

# 4. Plot the Red Dot at EER
plt.scatter(eer_fpr, eer_tpr, color='red', s=50, label=f'EER Point (FPR={eer_fpr:.2f})', zorder=4)

# Styling
plt.legend(loc='lower right', fontsize=10)
plt.xlabel('False Positive Rate (FPR)', fontsize=12)
plt.ylabel('True Positive Rate (TPR)', fontsize=12)
plt.grid(True, linestyle=':', alpha=0.6)
plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.02])

plt.tight_layout()
plt.savefig(
    os.path.join(utk_face_results_dir, 'ROC curve for UTKFace (Gender Recognition).png')
)
plt.close()

In [48]:
class_names = ['female', 'male']

In [49]:
plt.figure(figsize=(10, 8), dpi=800)
sns.heatmap(gender_cm, annot=True, fmt='.2f', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.ylabel('True Label', fontsize=12, labelpad = 30)
plt.xlabel('Predicted Label', fontsize=12, labelpad = 30)
plt.title('Confusion Matrix for UTKFace (Gender Recognition)', fontsize=14, pad = 30)
plt.savefig(
    os.path.join(utk_face_results_dir, 'Confusion Matrix UTKFace (Gender Recognition).png')
)
plt.close()

### MORPH

In [50]:
morph_results_dir = os.path.join(gender_recognition_results_dir, 'MORPH')
os.makedirs(morph_results_dir, exist_ok = True)

In [51]:
gender_accuracy, gender_f1, gender_precision, gender_recall, gender_cm, fpr, tpr, auc_score, thresholds = eval.evaluate_gender(model, morph_dataloader)

In [52]:
with open(os.path.join(morph_results_dir, 'results.txt'), 'w') as file:
    result_txt_output = f'Accuracy for MORPH (Gender Recognition): {gender_accuracy}\n'
    result_txt_output += f'F1 Score for MORPH (Gender Recognition): {gender_f1}\n'
    result_txt_output += f'AUC Score for MORPH (Gender Recognition): {auc_score}\n'
    result_txt_output += f'Precision for MORPH (Gender Recognition): {gender_precision}\n'
    result_txt_output += f'Recall for MORPH (Gender Recognition): {gender_recall}\n'
    file.write(result_txt_output)
print(result_txt_output)

Accuracy for MORPH (Gender Recognition): 0.9994002399040384
F1 Score for MORPH (Gender Recognition): 0.999648341343336
AUC Score for MORPH (Gender Recognition): 0.9999534998675065
Precision for MORPH (Gender Recognition): 0.9997655334114889
Recall for MORPH (Gender Recognition): 0.9995311767463666



In [53]:

# 1. Calculate the Equal Error Rate (EER) Point
# This is where False Accept Rate (fpr) ~ False Reject Rate (1 - tpr)
# This corresponds to the intersection with the diagonal y = 1 - x
fnr = 1 - tpr
eer_index = np.nanargmin(np.absolute((fnr - fpr)))
eer_fpr = fpr[eer_index]
eer_tpr = tpr[eer_index]

plt.figure(figsize=(8, 6), dpi = 800)
plt.title(f'ROC Curve for MORPH (Gender Recognition)', fontsize=14, fontweight='bold')

# 2. Plot the Random Model (Gray Dashed Line)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', alpha=0.7, label='Random Guessing')

# 3. Plot the Main Model (Royal Blue)
# Using 'zorder' to ensure it sits on top of the grid/random line
plt.plot(fpr, tpr, color='#4169E1', linewidth=2, label=f'AUC: {auc_score:.4f}', zorder=3)

# 4. Plot the Red Dot at EER
plt.scatter(eer_fpr, eer_tpr, color='red', s=50, label=f'EER Point (FPR={eer_fpr:.2f})', zorder=4)

# Styling
plt.legend(loc='lower right', fontsize=10)
plt.xlabel('False Positive Rate (FPR)', fontsize=12)
plt.ylabel('True Positive Rate (TPR)', fontsize=12)
plt.grid(True, linestyle=':', alpha=0.6)
plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.02])

plt.tight_layout()
plt.savefig(
    os.path.join(morph_results_dir, 'ROC curve for MORPH (Gender Recognition).png')
)
plt.close()

In [54]:
plt.figure(figsize=(10, 8), dpi=800)
sns.heatmap(gender_cm, annot=True, fmt='.2f', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.ylabel('True Label', fontsize=12, labelpad = 30)
plt.xlabel('Predicted Label', fontsize=12, labelpad = 30)
plt.title('Confusion Matrix for MORPH (Gender Recognition)', fontsize=14, pad = 30)
plt.savefig(
    os.path.join(morph_results_dir, 'Confusion Matrix MORPH (Gender Recognition).png')
)
plt.close()

### FairFace

In [13]:
fairface_db = datasets.FairFace(subset = 'test', transform = test_transform)

In [14]:
fairface_dataloader = torch.utils.data.DataLoader(
    dataset = fairface_db,
    batch_size = 32,
    num_workers =2
)

In [57]:
fairface_results_dir = os.path.join(gender_recognition_results_dir, 'FairFace')
os.makedirs(fairface_results_dir, exist_ok = True)

In [58]:
gender_accuracy, gender_f1, gender_precision, gender_recall, gender_cm, fpr, tpr, auc_score, thresholds = eval.evaluate_gender(model, fairface_dataloader)

In [59]:
with open(os.path.join(fairface_results_dir, 'results.txt'), 'w') as file:
    result_txt_output = f'Accuracy for FairFace (Gender Recognition): {gender_accuracy}\n'
    result_txt_output += f'F1 Score for FairFace (Gender Recognition): {gender_f1}\n'
    result_txt_output += f'AUC Score for FairFace (Gender Recognition): {auc_score}\n'
    result_txt_output += f'Precision for FairFace (Gender Recognition): {gender_precision}\n'
    result_txt_output += f'Recall for FairFace (Gender Recognition): {gender_recall}\n'
    file.write(result_txt_output)
print(result_txt_output)

Accuracy for FairFace (Gender Recognition): 0.9595581522731422
F1 Score for FairFace (Gender Recognition): 0.961747690182195
AUC Score for FairFace (Gender Recognition): 0.9889711804388637
Precision for FairFace (Gender Recognition): 0.96199689065469
Recall for FairFace (Gender Recognition): 0.9614986187845304



In [60]:

# 1. Calculate the Equal Error Rate (EER) Point
# This is where False Accept Rate (fpr) ~ False Reject Rate (1 - tpr)
# This corresponds to the intersection with the diagonal y = 1 - x
fnr = 1 - tpr
eer_index = np.nanargmin(np.absolute((fnr - fpr)))
eer_fpr = fpr[eer_index]
eer_tpr = tpr[eer_index]

plt.figure(figsize=(8, 6), dpi = 800)
plt.title(f'ROC Curve for FairFace (Gender Recognition)', fontsize=14, fontweight='bold')

# 2. Plot the Random Model (Gray Dashed Line)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', alpha=0.7, label='Random Guessing')

# 3. Plot the Main Model (Royal Blue)
# Using 'zorder' to ensure it sits on top of the grid/random line
plt.plot(fpr, tpr, color='#4169E1', linewidth=2, label=f'AUC: {auc_score:.4f}', zorder=3)

# 4. Plot the Red Dot at EER
plt.scatter(eer_fpr, eer_tpr, color='red', s=50, label=f'EER Point (FPR={eer_fpr:.2f})', zorder=4)

# Styling
plt.legend(loc='lower right', fontsize=10)
plt.xlabel('False Positive Rate (FPR)', fontsize=12)
plt.ylabel('True Positive Rate (TPR)', fontsize=12)
plt.grid(True, linestyle=':', alpha=0.6)
plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.02])

plt.tight_layout()
plt.savefig(
    os.path.join(fairface_results_dir, 'ROC curve for FairFace (Gender Recognition).png')
)
plt.close()

In [61]:
plt.figure(figsize=(10, 8), dpi=800)
sns.heatmap(gender_cm, annot=True, fmt='.2f', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.ylabel('True Label', fontsize=12, labelpad = 30)
plt.xlabel('Predicted Label', fontsize=12, labelpad = 30)
plt.title('Confusion Matrix for FairFace (Gender Recognition)', fontsize=14, pad = 30)
plt.savefig(
    os.path.join(fairface_results_dir, 'Confusion Matrix FairFace (Gender Recognition).png')
)
plt.close()

### AgeDB

In [62]:
agedb_results_dir = os.path.join(gender_recognition_results_dir, 'AgeDB')
os.makedirs(agedb_results_dir, exist_ok = True)

In [63]:
gender_accuracy, gender_f1, gender_precision, gender_recall, gender_cm, fpr, tpr, auc_score, thresholds = eval.evaluate_gender(model, agedb_dataloader)

In [64]:
with open(os.path.join(agedb_results_dir, 'results.txt'), 'w') as file:
    result_txt_output = f'Accuracy for AgeDB (Gender Recognition): {gender_accuracy}\n'
    result_txt_output += f'F1 Score for AgeDB (Gender Recognition): {gender_f1}\n'
    result_txt_output += f'AUC Score for AgeDB (Gender Recognition): {auc_score}\n'
    result_txt_output += f'Precision for AgeDB (Gender Recognition): {gender_precision}\n'
    result_txt_output += f'Recall for AgeDB (Gender Recognition): {gender_recall}\n'
    file.write(result_txt_output)
print(result_txt_output)

Accuracy for AgeDB (Gender Recognition): 0.9762857836001941
F1 Score for AgeDB (Gender Recognition): 0.9800071585621517
AUC Score for AgeDB (Gender Recognition): 0.9862718360587743
Precision for AgeDB (Gender Recognition): 0.9809601801617361
Recall for AgeDB (Gender Recognition): 0.9790559869227625



In [65]:

# 1. Calculate the Equal Error Rate (EER) Point
# This is where False Accept Rate (fpr) ~ False Reject Rate (1 - tpr)
# This corresponds to the intersection with the diagonal y = 1 - x
fnr = 1 - tpr
eer_index = np.nanargmin(np.absolute((fnr - fpr)))
eer_fpr = fpr[eer_index]
eer_tpr = tpr[eer_index]

plt.figure(figsize=(8, 6), dpi = 800)
plt.title(f'ROC Curve for AgeDB (Gender Recognition)', fontsize=14, fontweight='bold')

# 2. Plot the Random Model (Gray Dashed Line)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', alpha=0.7, label='Random Guessing')

# 3. Plot the Main Model (Royal Blue)
# Using 'zorder' to ensure it sits on top of the grid/random line
plt.plot(fpr, tpr, color='#4169E1', linewidth=2, label=f'AUC: {auc_score:.4f}', zorder=3)

# 4. Plot the Red Dot at EER
plt.scatter(eer_fpr, eer_tpr, color='red', s=50, label=f'EER Point (FPR={eer_fpr:.2f})', zorder=4)

# Styling
plt.legend(loc='lower right', fontsize=10)
plt.xlabel('False Positive Rate (FPR)', fontsize=12)
plt.ylabel('True Positive Rate (TPR)', fontsize=12)
plt.grid(True, linestyle=':', alpha=0.6)
plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.02])

plt.tight_layout()
plt.savefig(
    os.path.join(agedb_results_dir, 'ROC curve for AgeDB (Gender Recognition).png')
)
plt.close()

In [66]:
plt.figure(figsize=(10, 8), dpi=800)
sns.heatmap(gender_cm, annot=True, fmt='.2f', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.ylabel('True Label', fontsize=12, labelpad = 30)
plt.xlabel('Predicted Label', fontsize=12, labelpad = 30)
plt.title('Confusion Matrix for AgeDB (Gender Recognition)', fontsize=14, pad = 30)
plt.savefig(
    os.path.join(agedb_results_dir, 'Confusion Matrix AgeDB (Gender Recognition).png')
)
plt.close()

## Race Recognition

### UTKFace

In [67]:
race_result_dir = os.path.join(results_dir, "Race Recognition")
os.makedirs(race_result_dir, exist_ok=True)

In [68]:
utk_face_results_dir = os.path.join(race_result_dir, 'UTKFace')
os.makedirs(utk_face_results_dir, exist_ok=True)

In [69]:
race_accuracy, race_f1, race_precision, race_recall, race_cm = eval.evaluate_race(model, utk_face_dataloader, device = 'cuda')

Validating Race:   0%|          | 0/75 [00:00<?, ?it/s]

In [70]:
with open(os.path.join(utk_face_results_dir, 'results.txt'), 'w') as file:
    result_txt_output = f'Accuracy for UTKFace (Race Recognition): {race_accuracy}\n'
    result_txt_output += f'F1 Score for UTKFace (Race Recognition): {race_f1}\n'
    result_txt_output += f'Precision for UTKFace (Race Recognition): {race_precision}\n'
    result_txt_output += f'Recall for UTKFace (Race Recognition): {race_recall}\n'
    file.write(result_txt_output)
print(result_txt_output)

Accuracy for UTKFace (Race Recognition): 0.8775854791051076
F1 Score for UTKFace (Race Recognition): 0.8784843626813051
Precision for UTKFace (Race Recognition): 0.8796921677730632
Recall for UTKFace (Race Recognition): 0.8775854791051076



In [71]:
class_names = ['White', 'Black', 'Asian', 'Indian', 'Other']

In [72]:
plt.figure(figsize=(10, 8), dpi=800)
sns.heatmap(race_cm, annot=True, fmt='.2f', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.ylabel('True Label', fontsize=12, labelpad = 30)
plt.xlabel('Predicted Label', fontsize=12, labelpad = 30)
plt.title('Confusion Matrix for UTKFace (Race Recognition)', fontsize=14, pad = 30)
plt.savefig(
    os.path.join(utk_face_results_dir, 'Confusion Matrix UTKFace (Race Recognition).png')
)
plt.close()

### FairFace

In [73]:
fairface_results_dir = os.path.join(race_result_dir, 'FairFace')
os.makedirs(fairface_results_dir, exist_ok=True)

In [74]:
race_accuracy, race_f1, race_precision, race_recall, race_cm = eval.evaluate_race(model, fairface_dataloader, device = 'cuda')

Validating Race:   0%|          | 0/343 [00:00<?, ?it/s]

In [75]:
with open(os.path.join(fairface_results_dir, 'results.txt'), 'w') as file:
    result_txt_output = f'Accuracy for FairFace (Race Recognition): {race_accuracy}\n'
    result_txt_output += f'F1 Score for FairFace (Race Recognition): {race_f1}\n'
    result_txt_output += f'Precision for FairFace (Race Recognition): {race_precision}\n'
    result_txt_output += f'Recall for FairFace (Race Recognition): {race_recall}\n'
    file.write(result_txt_output)
print(result_txt_output)

Accuracy for FairFace (Race Recognition): 0.8148621508124886
F1 Score for FairFace (Race Recognition): 0.8146548517844996
Precision for FairFace (Race Recognition): 0.8146694553946627
Recall for FairFace (Race Recognition): 0.8148621508124886



In [76]:
plt.figure(figsize=(10, 8), dpi=800)
sns.heatmap(race_cm, annot=True, fmt='.2f', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.ylabel('True Label', fontsize=12, labelpad = 30)
plt.xlabel('Predicted Label', fontsize=12, labelpad = 30)
plt.title('Confusion Matrix for FairFace (Race Recognition)', fontsize=14, pad = 30)
plt.savefig(
    os.path.join(fairface_results_dir, 'Confusion Matrix FairFace (Race Recognition).png')
)
plt.close()

## Attribute recognition

In [19]:
attribute_result_dir = os.path.join(results_dir, 'attribute recognition')
os.makedirs(attribute_result_dir, exist_ok = True)

In [20]:
attribute_test_transform = v2.Compose([
    v2.ToPILImage(),
    v2.Resize(image_size, antialias=True),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale = True),
    v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

In [21]:
celeba_result_dir = os.path.join(attribute_result_dir, 'CelebA')
os.makedirs(celeba_result_dir, exist_ok = True)

In [22]:
celeba_validation_db = torch.utils.data.DataLoader(
    datasets.CelebA(subset = 'validation', transform = attribute_test_transform),
    batch_size = 32,
    num_workers = 2
)

celeba_test_db = torch.utils.data.DataLoader(
    datasets.CelebA(subset = 'test', transform = attribute_test_transform),
    batch_size = 32,
    num_workers = 2
)

In [81]:
thresholds = np.full(40, 0.5)

# Attribute recognition
element_wise_accuracy, attribute_f1, attribute_precision, attribute_recall, thresholds_to_use = eval.evaluate_attributes(model = model, dataloader = celeba_validation_db, thresholds=thresholds)
print(f'Accuracy for CelebA (attribute recognition) validation = {element_wise_accuracy}.')
print(f'F1 score for CelebA (attribute recognition) validation = {attribute_f1}')

element_wise_accuracy, attribute_f1, attribute_precision, attribute_recall, thresholds_to_use = eval.evaluate_attributes(model = model, dataloader = celeba_test_db, thresholds=thresholds)
print(f'Accuracy for CelebA (attribute recognition) test = {element_wise_accuracy}.')
print(f'F1 score for CelebA (attribute recognition) test = {attribute_f1}')

Accuracy for CelebA (attribute recognition) validation = 0.9293942215734635.
F1 score for CelebA (attribute recognition) validation = 0.8357873479959612


Accuracy for CelebA (attribute recognition) test = 0.9178501653140968.
F1 score for CelebA (attribute recognition) test = 0.8136627436275471


In [82]:
with open(os.path.join(celeba_result_dir, 'results.txt'), 'w') as file:
    result_txt_output = f'Accuracy for CelebA (Attribute Estimation): {element_wise_accuracy}\n'
    result_txt_output += f'F1 Score for CelebA (Attribute Estimation): {attribute_f1}\n'
    result_txt_output += f'Precision for CelebA (Attribute Estimation): {attribute_precision}\n'
    result_txt_output += f'Recall for CelebA (Attribute Estimation): {attribute_recall}\n'
    file.write(result_txt_output)
print(result_txt_output)

Accuracy for CelebA (Attribute Estimation): 0.9178501653140968
F1 Score for CelebA (Attribute Estimation): 0.8136627436275471
Precision for CelebA (Attribute Estimation): 0.8556730596881161
Recall for CelebA (Attribute Estimation): 0.7755844746632874



## Head pose

In [83]:
head_pose_result_dir = os.path.join(results_dir, 'head pose')

In [84]:
biwi_results_dir = os.path.join(head_pose_result_dir, 'BIWI')
os.makedirs(biwi_results_dir, exist_ok = True)

In [85]:
biwi_dataloader = torch.utils.data.DataLoader(
    datasets.BIWI(transform = test_transform),
    batch_size = 32,
    num_workers = 2
)

In [86]:
mae_mean, mae_roll, mae_pitch, mae_yaw = eval.evaluate_head_pose(model, biwi_dataloader)

In [87]:
with open(os.path.join(biwi_results_dir, 'results.txt'), 'w') as file:
    result_txt_output = f'MAE for BIWI (Pose Estimation): {mae_mean}\n'
    result_txt_output += f'Roll MAE Score for BIWI (Pose Estimation): {mae_roll}\n'
    result_txt_output += f'Pitch for BIWI (Pose Estimation): {mae_pitch}\n'
    result_txt_output += f'Yaw for BIWI (Pose Estimation): {mae_yaw}\n'
    file.write(result_txt_output)
print(result_txt_output)

MAE for BIWI (Pose Estimation): 4.60022497177124
Roll MAE Score for BIWI (Pose Estimation): 3.2856862545013428
Pitch for BIWI (Pose Estimation): 6.015726566314697
Yaw for BIWI (Pose Estimation): 4.499261379241943



# Landmarks 98

In [88]:
landmarks_98_results_dir = os.path.join(results_dir, 'landmarks 98')
os.makedirs(landmarks_98_results_dir, exist_ok = True)

In [89]:
wflw_validation_db = torch.utils.data.DataLoader( # for landmark 98 detection
    dataset = datasets.WFLW(transform = test_transform, subset = 'test'),
    batch_size = 32,
    shuffle = False,
    num_workers = 2,
    pin_memory = True,
)

In [90]:
pixel_mae_98, nme_98 = eval.evaluate_landmarks_98(model = model, dataloader = wflw_validation_db)
print(f'WFLW: Pixel MAE: {pixel_mae_98}, NME: {nme_98}  (WFLW Landmark 98 detection)')


WFLW: Pixel MAE: 1.7418839931488037, NME: 4.384979248046875  (WFLW Landmark 98 detection)


In [91]:
with open(os.path.join(landmarks_98_results_dir, 'results.txt'), 'w') as file:
    result_txt_output = f'Pixel MAE for WFLW (Landmark 98 Detection): {pixel_mae_98}\n'
    result_txt_output += f'NME for WFLW (Landmark 98 Detection): {nme_98}\n'
    file.write(result_txt_output)

# Landmarks 68

In [92]:
landmarks_68_results_dir = os.path.join(results_dir, 'landmarks 68')
os.makedirs(landmarks_68_results_dir, exist_ok = True)

In [93]:
w300_validation_db_full = torch.utils.data.DataLoader( # for landmark detection
    dataset = datasets.W300(transform = test_transform, subset = 'full'),
    batch_size = 32,
    shuffle = False,
    num_workers = 2,
    pin_memory = True,
)

w300_validation_db_common = torch.utils.data.DataLoader( # for landmark detection
    dataset = datasets.W300(transform = test_transform, subset = 'common'),
    batch_size = 32,
    shuffle = False,
    num_workers = 2,
    pin_memory = True,
)

w300_validation_db_challenge = torch.utils.data.DataLoader( # for landmark detection
    dataset = datasets.W300(transform = test_transform, subset = 'challenge'),
    batch_size = 32,
    shuffle = False,
    num_workers = 2,
    pin_memory = True,
)

In [94]:
# Landmark detection 68
pixel_mae_full, nme_full = eval.evaluate_landmarks_68(model = model, dataloader = w300_validation_db_full, image_size = image_size[0])
print(f'300W Full: Pixel MAE: {pixel_mae_full}, NME: {nme_full}  (W300 Landmark detection)')

pixel_mae_common, nme_common = eval.evaluate_landmarks_68(model = model, dataloader = w300_validation_db_common, image_size = image_size[0])
print(f'300W Common: Pixel MAE: {pixel_mae_common}, NME: {nme_common}  (W300 Landmark detection)')

pixel_mae_challenge, nme_challenge = eval.evaluate_landmarks_68(model = model, dataloader = w300_validation_db_challenge, image_size = image_size[0])
print(f'300W Challenge: Pixel MAE: {pixel_mae_challenge}, NME: {nme_challenge}  (W300 Landmark detection)')


300W Full: Pixel MAE: 1.5258530378341675, NME: 3.563446283340454  (W300 Landmark detection)


300W Common: Pixel MAE: 1.374380350112915, NME: 3.115673303604126  (W300 Landmark detection)


300W Challenge: Pixel MAE: 2.1474521160125732, NME: 5.400973320007324  (W300 Landmark detection)


In [95]:
with open(os.path.join(landmarks_68_results_dir, 'results.txt'), 'w') as file:
    result_txt_output = f'Pixel MAE for W300 (Landmark 68 Detection): {pixel_mae_full}\n'
    result_txt_output += f'NME for W300 (Landmark 68 Detection): {nme_full}\n'
    result_txt_output += f'Pixel MAE for W300 Common (Landmark 68 Detection): {pixel_mae_common}\n'
    result_txt_output += f'NME for W300 Common (Landmark 68 Detection): {nme_common}\n'
    result_txt_output += f'Pixel MAE for W300 Challenge (Landmark 68 Detection): {pixel_mae_challenge}\n'
    result_txt_output += f'NME for W300 Challenge (Landmark 68 Detection): {nme_challenge}\n'
    file.write(result_txt_output)

# Visibility Detection

In [96]:
landmarks_29_visibility_results_dir = os.path.join(results_dir, 'landmarks 29 visibility')
os.makedirs(landmarks_29_visibility_results_dir, exist_ok = True)

In [97]:
cofw_validation_db = torch.utils.data.DataLoader( # for landmark 29 detection and visibility
    dataset = datasets.COFW(transform = test_transform, subset = 'test'),
    batch_size = 32,
    shuffle = False,
    num_workers = 2,
    pin_memory = True,
)

In [98]:
# Visibility detection
acc_50, f1_50, prec_50, rec_50, cm_50, recall_at_80_prec, best_threshold = eval.evaluate_visibility_29(model = model, dataloader = cofw_validation_db)
print(f'Visibility Prediction (COFW): Accuracy: {acc_50}, F1: {f1_50}, Precision: {prec_50}, Recall: {rec_50}, Recall at 80% Precision: {recall_at_80_prec}, Best Threshold: {best_threshold}')


Visibility Prediction (COFW): Accuracy: 0.9128069101543903, F1: 0.7926237463604011, Precision: 0.8687943262411347, Recall: 0.7287328970850684, Recall at 80% Precision: 0.8158834027364664, Best Threshold: 0.0008453094051219523


In [99]:
with open(os.path.join(landmarks_29_visibility_results_dir, 'results.txt'), 'w') as file:
    result_txt_output = f'Accuracy for COFW (Visibility Detection): {acc_50}\n'
    result_txt_output += f'F1 Score for COFW (Visibility Detection): {f1_50}\n'
    result_txt_output += f'Precision for COFW (Visibility Detection): {prec_50}\n'
    result_txt_output += f'Recall for COFW (Visibility Detection): {rec_50}\n'
    result_txt_output += f'Recall at 80% Precision for COFW (Visibility Detection): {recall_at_80_prec}\n'
    result_txt_output += f'Best Threshold for COFW (Visibility Detection): {best_threshold}\n'
    file.write(result_txt_output)

## Bias analysis

In [15]:
bias_results_dir = os.path.join(results_dir, 'bias evaluation')
os.makedirs(bias_results_dir, exist_ok = True)

In [16]:
result = eval.evaluate_bias_fairface(model, fairface_dataloader, device = 'cuda')


 FAIRFACE BIAS AND FAIRNESS EVALUATION
White Gender Acc:     96.26%
Non-White Gender Acc: 95.88%
Gender Discrepancy:   0.37
--------------------------------------------------
White Age Acc:        55.25%
Non-White Age Acc:    54.78%
Age Discrepancy:      0.48



In [17]:
with open(os.path.join(bias_results_dir, 'results.txt'), 'w') as file:
    result_txt_output = f'Bias Evaluation Results (FairFace):\n'
    for key, value in result.items():
        result_txt_output += f'{key}: {value}\n'
    file.write(result_txt_output)
print(result_txt_output)

Bias Evaluation Results (FairFace):
white_gender_acc: 96.2589928057554
nonwhite_gender_acc: 95.88454166196865
gender_discrepancy: 0.37445114378674305
white_age_acc: 55.25179856115108
nonwhite_age_acc: 54.775059194948696
age_discrepancy: 0.47673936620238067



In [23]:
image_names = ['148327.jpg', '129195.jpg', '157190.jpg', '202577.jpg', '202586.jpg', '202579.jpg']
cropping_side_length = int(23 / 112 * image_size[0])  # 23 pixels in the original 112x112 image corresponds to the cropping side length

rgb_imgs = []
for image_name in image_names:
    image_path = os.path.join('data', 'datasets', 'attribute recognition', 'CelebA', 'Images', image_name)
    rgb_img = cv2.imread(image_path, 1)[:, :, ::-1]
    rgb_img = cv2.resize(rgb_img, image_size)
    rgb_img_float = np.float32(rgb_img) / 255.0
    rgb_imgs.append(rgb_img_float)
    
input_tensors = []
for rgb_img in rgb_imgs:
    input_tensor = torch.reshape(attribute_test_transform(rgb_img), (1, 3, image_size[0], image_size[1])).to('cuda').requires_grad_(True)
    input_tensors.append(input_tensor)

for i in range(len(rgb_imgs)):
    rgb_imgs[i] = (input_tensors[i].detach().cpu().squeeze().permute(1, 2, 0).numpy() + 1)/2

In [24]:
import torch.nn.functional as F

tokens_per_task = {
    'emotion_recognition' : 3,
    'age_estimation' : 1,
    'gender_recognition' : 1,
    'race_recognition' : 1,
    'Whole' : 3,          #
    'Hair' : 3,           #
    'Eyes' : 2,           #
    'Nose' : 2,           #     Attribute estimation
    'Cheek' : 2,          #
    'Mouth' : 3,          #
    'Neck' : 2,           #
    'head_pose_estimation' : 1,
    'landmarks_98' : 3,
    'landmarks_68' : 3,
    'landmarks_29_visibility': 3
}
split_sizes = list(tokens_per_task.values())
keys = list(tokens_per_task.keys())

In [25]:
from utility_scripts.pose_estimation_utilities import compute_rotation_matrix_from_ortho6d, matrix_to_euler_angles, draw_axis
from utility_scripts.age_estimation_utilities import get_age_expectation

In [26]:
def get_attention_maps_and_predictions(model, input_tensor, split_sizes):
    model.to('cuda')
    model.eval()
    with torch.no_grad():
        output = model(input_tensor, return_attention=True)
        attention_maps = output[-1]

        per_layer_attention_maps = []
        for i in range(len(attention_maps)):
            attention_maps[i] = attention_maps[i].mean(axis = 1).squeeze(0) # average across all heads
            split_tensors = torch.split(attention_maps[i], split_sizes, dim = 0)
            per_task_attention_maps = []
            for j, split_tensor in enumerate(split_tensors):
                stage_0 = split_tensor[:, 0:56*56].reshape(-1, 56, 56)
                stage_1 = split_tensor[:, 56*56:56*56 + 28*28].reshape(-1, 28, 28)
                stage_2 = split_tensor[:, 56*56 + 28*28:56*56 + 28*28 + 14*14].reshape(-1, 14, 14)
                stage_3 = split_tensor[:, 56*56 + 28*28 + 14*14:56*56 + 28*28 + 14*14 + 7*7].reshape(-1, 7, 7)

                stage_0_resized = F.interpolate(stage_0.unsqueeze(0), size=image_size, mode='bicubic').squeeze(0).permute(1, 2, 0).detach().cpu().numpy()
                stage_1_resized = F.interpolate(stage_1.unsqueeze(0), size=image_size, mode='bicubic').squeeze(0).permute(1, 2, 0).detach().cpu().numpy()
                stage_2_resized = F.interpolate(stage_2.unsqueeze(0), size=image_size, mode='bicubic').squeeze(0).permute(1, 2, 0).detach().cpu().numpy()
                stage_3_resized = F.interpolate(stage_3.unsqueeze(0), size=image_size, mode='bicubic').squeeze(0).permute(1, 2, 0).detach().cpu().numpy()

                
                combined_attention_map = (stage_0_resized + stage_1_resized + stage_2_resized + stage_3_resized) / 4.0
                combined_attention_map = combined_attention_map.mean(axis = 2) # average across all tokens for the task to get a single attention map per image
                combined_attention_map_normalized = (combined_attention_map - np.min(combined_attention_map)) / (np.max(combined_attention_map) - np.min(combined_attention_map) + 1e-8)
                per_task_attention_maps.append(combined_attention_map_normalized)
            per_layer_attention_maps.append(per_task_attention_maps)
        
        per_layer_attention_maps = np.array(per_layer_attention_maps) # Layers, Tasks, H, W

    per_task_attention_maps = np.mean(per_layer_attention_maps, axis = 0) # Average across layers
    return per_task_attention_maps, output[:-1]


def get_predictions_from_model_output(model_output):
    
        (normalized_embedding, embedding_norm), emotion_recognition_output, age_estimation_output, gender_output, race_recognition_output, attribute_recognition_output, head_pose_estimation_output, landmarks_98_output, landmarks_68_output, landmarks_29_vis_output, visibility_output = model_output
        
        # emotion_recognition
        emotion_label_translation = {
        0 : 'Neutral',
        1 : 'Happy',
        2 : 'Sad',
        3 : 'Surprised',
        4 : 'Afraid',
        5 : 'Disgusted',
        6 : 'Angry'
        }
        
        emotion_prediction = emotion_label_translation[torch.argmax(emotion_recognition_output, dim = 1).item()]
        
        age_prediction = get_age_expectation(age_estimation_output)
        
        gender_prediction = "Male" if int(torch.sigmoid(gender_output) > 0.5) else "Female"
        
        # race recognition
        race_label_translation = {
            0 : 'White',
            1 : 'Black',
            2 : 'Asian',
            3 : 'Indian',
            4 : 'Other'
        }
        
        race_prediction = race_label_translation[torch.argmax(race_recognition_output, dim=1).item()]
        
        # attribute recognition
        attribute_groups = {
                'Whole' : ['Attractive', 'Blurry', 'Chubby', 'Heavy_Makeup', 'Oval_Face', 'Pale_Skin', 'Male', 'Young'],
                'Hair' : ['Bald', 'Bangs', 'Black_Hair', 'Blond_Hair', 'Brown_Hair', 'Gray_Hair', 'Receding_Hairline', 'Straight_Hair', 'Wavy_Hair', 'Wearing_Hat'],
                'Eyes' : ['Arched_Eyebrows', 'Bags_Under_Eyes', 'Bushy_Eyebrows', 'Eyeglasses', 'Narrow_Eyes'],
                'Nose' : ['Big_Nose', 'Pointy_Nose'],
                'Cheek' : ['High_Cheekbones', 'Rosy_Cheeks', 'Wearing_Earrings', 'Sideburns'],
                'Mouth' : ['5_o_Clock_Shadow', 'Big_Lips', 'Mouth_Slightly_Open', 'Mustache', 'Wearing_Lipstick', 'No_Beard', 'Smiling', 'Goatee'],
                'Neck' : ['Wearing_Necklace', 'Wearing_Necktie', 'Double_Chin']
        }

        attribute_list = []
        for group_name in attribute_groups.keys():
            attributes = attribute_groups[group_name]
            attribute_list = attribute_list + attributes

        attribute_list = np.array(attribute_list)
        
        attribute_prediction = attribute_list[(torch.sigmoid(attribute_recognition_output) > 0.5).squeeze(0).cpu().numpy()]

        
        euler_angles = matrix_to_euler_angles(compute_rotation_matrix_from_ortho6d(head_pose_estimation_output))
        roll_prediction = euler_angles[0][0]
        pitch_prediction = euler_angles[0][1]
        yaw_prediction = euler_angles[0][2]
        
        landmark_98_predictions = landmarks_98_output.squeeze(0).detach().cpu().numpy()
        landmark_68_predictions = landmarks_68_output.squeeze(0).detach().cpu().numpy()
        landmark_29_visibility_predictions = landmarks_29_vis_output.squeeze(0).detach().cpu().numpy()
        visibility_output = torch.sigmoid(visibility_output).squeeze(0).detach().cpu().numpy()
        
        return emotion_prediction, age_prediction, gender_prediction, race_prediction, attribute_prediction, (roll_prediction, pitch_prediction, yaw_prediction), landmark_98_predictions, landmark_68_predictions, landmark_29_visibility_predictions, visibility_output
        
def get_attention_visualization_from_attribute_estimation_image(model, input_tensor, split_sizes, cropping_side_length = 23):
    per_task_attention_maps_attributes, attribute_predictions = get_attention_maps_and_predictions(model, input_tensor, split_sizes)
    per_task_attention_maps_no_attributes, task_predictions = get_attention_maps_and_predictions(model, torchvision.transforms.Resize(image_size)(input_tensor[:, :, cropping_side_length:-cropping_side_length, cropping_side_length:-cropping_side_length]), split_sizes)

    per_task_attention_maps_attributes = per_task_attention_maps_attributes[4:11] # Only the attribute estimation tasks from the uncropped image
    per_task_attention_maps_final = np.concatenate((per_task_attention_maps_no_attributes[0:4], np.mean(per_task_attention_maps_attributes, axis = 0, keepdims = True), per_task_attention_maps_no_attributes[11:]), axis = 0) # All tasks except attribute estimation
    
    rgb_img = (input_tensor.detach().cpu().squeeze().permute(1, 2, 0).numpy() + 1)/2
    cropped_rgb_img = (torchvision.transforms.Resize(image_size)(input_tensor[:, :, cropping_side_length:-cropping_side_length, cropping_side_length:-cropping_side_length]).detach().cpu().squeeze().permute(1, 2, 0).numpy() + 1)/2
    
    visualization_images = []
    for i in range(len(per_task_attention_maps_final)):
        if i != 4: # Skip the attribute estimation task
            visualization_image = show_cam_on_image(cropped_rgb_img, per_task_attention_maps_final[i], use_rgb = True, image_weight = 0.5)
            visualization_images.append(visualization_image)
        else:
            visualization_image = show_cam_on_image(rgb_img, per_task_attention_maps_final[i], use_rgb = True, image_weight = 0.5)
            visualization_images.append(visualization_image)
            
    attribute_predictions = attribute_predictions[5]
    task_predictions = task_predictions[0:5] + (attribute_predictions, ) + task_predictions[6:]
    
    return np.array(visualization_images), get_predictions_from_model_output(task_predictions)


def draw_landmarks_on_image(image, landmarks, visibility = None, color_visible = (0, 255, 0), color_invisible = (255, 0, 0), radius = 1, image_size = 112):

    for i in range(landmarks.shape[0] // 2):
        x, y = int(landmarks[2 * i] * image_size) , int(landmarks[2 * i + 1] * image_size) 
        x = np.clip(x, 0, image_size - 1)
        y = np.clip(y, 0, image_size - 1)
        
        if visibility is not None:
            if visibility[i] < 0.5:
                cv2.circle(image, (x, y), radius, color_visible, 1)
            else:
                cv2.circle(image, (x, y), radius, color_invisible, 1)
        else:
            cv2.circle(image, (x, y), radius, color_visible, 1)
    return image

def get_landmark_and_visibility_visualization_images(rgb_image, landmark_98_predictions, landmark_68_predictions, landmark_29_visibility_predictions, visibility_output):    
    landmark_98_img = draw_landmarks_on_image(rgb_image.copy(), landmark_98_predictions, image_size = image_size[0])
    landmark_68_img = draw_landmarks_on_image(rgb_image.copy(), landmark_68_predictions, image_size = image_size[0])
    landmark_29_visibility_img = draw_landmarks_on_image(rgb_image.copy(), landmark_29_visibility_predictions, visibility=visibility_output, image_size = image_size[0])

    return landmark_98_img, landmark_68_img, landmark_29_visibility_img
    

In [27]:
image_index = 3

In [28]:
def get_image_visualization(model, input_tensor, split_sizes, cropping_side_length = 23, figure_name = None):
    rgb_image = (input_tensor.detach().cpu().squeeze().permute(1, 2, 0).numpy() + 1)/2
    cropped_rgb_image = cv2.resize(rgb_image[cropping_side_length:-cropping_side_length, cropping_side_length:-cropping_side_length, :], image_size)
    visualization_images, predictions = get_attention_visualization_from_attribute_estimation_image(model, input_tensor, split_sizes, cropping_side_length = cropping_side_length)
    emotion_prediction, age_prediction, gender_prediction, race_prediction, attribute_prediction, (roll_prediction, pitch_prediction, yaw_prediction), landmark_98_predictions, landmark_68_predictions, landmark_29_visibility_predictions, visibility_output = predictions
    landmark_98_img, landmark_68_img, landmark_29_visibility_img = get_landmark_and_visibility_visualization_images(cropped_rgb_image, landmark_98_predictions, landmark_68_predictions, landmark_29_visibility_predictions, visibility_output)
    
    pad = 0
    fontsize = 20
    plt.figure(figsize = (24, 22), dpi=100)

    # Reference Image
    plt.subplot(4, 2, 1)
    plt.imshow(rgb_image)
    plt.axis('off')
    plt.title(f'Reference Image', fontsize=fontsize, pad = pad)

    # Emotion Recognition
    plt.subplot(4, 2, 2)
    plt.imshow(visualization_images[0])
    plt.axis('off')
    plt.title(f'Emotion Recognition\n Prediciton: {emotion_prediction}', fontsize=fontsize, pad = pad)

    # Age Estimation
    plt.subplot(4, 4, 5)
    plt.imshow(visualization_images[1])
    plt.axis('off')
    plt.title(f'Age Estimation\n Prediction: {age_prediction[0]:.2f}', fontsize=fontsize, pad = pad)

    # Gender Recognition
    plt.subplot(4, 4, 6)
    plt.imshow(visualization_images[2])
    plt.axis('off')
    plt.title(f'Gender Recognition\n Prediction: {gender_prediction}', fontsize=fontsize, pad = pad)

    # Race Recognition
    plt.subplot(4, 4, 7)
    plt.imshow(visualization_images[3])
    plt.axis('off')
    plt.title(f'Race Recognition\n Prediction: {race_prediction}', fontsize=fontsize, pad = pad)

    # Attribute Estimation
    plt.subplot(4, 4, 8)
    plt.imshow(visualization_images[4])
    plt.axis('off')

    counter = 0
    attribute_text = ""
    for i, attribute in enumerate(attribute_prediction):
        if i % 2 == 0 and i != 0:
            attribute_text += "\n"
        attribute_text += f"{attribute}, "
        counter += 1
        if counter >= 4:  # Limit to 4 attributes for display
            break

    attribute_text += '...'
    plt.title(f'Attribute Estimation\n Prediction: {attribute_text}', fontsize=fontsize, pad = pad)


    # Landmarks 98
    plt.subplot(4, 4, 9)
    plt.imshow(visualization_images[6])
    plt.axis('off')
    plt.title(f'Landmarks 98', fontsize=fontsize, pad = pad)

    # Landmarks 68
    plt.subplot(4, 4, 10)
    plt.imshow(visualization_images[7])
    plt.axis('off')
    plt.title(f'Landmarks 68', fontsize=fontsize, pad = pad)

    # Landmarks 29
    plt.subplot(4, 4, 11)
    plt.imshow(visualization_images[8])
    plt.axis('off')
    plt.title(f'Landmarks 29 and visibility', fontsize=fontsize, pad = pad)

    # Head Pose Estimation
    plt.subplot(4, 4, 12)
    plt.imshow(visualization_images[5])
    plt.axis('off')
    plt.title(f'Head Pose Estimation', fontsize=fontsize, pad = pad)

    # Landmarks 98 visualization
    plt.subplot(4, 4, 13)
    plt.imshow(landmark_98_img)
    plt.axis('off')
    plt.title(f'Landmarks 98', fontsize=fontsize, pad = pad)

    # Landmarks 68 visualization
    plt.subplot(4, 4, 14)
    plt.imshow(landmark_68_img)
    plt.axis('off')
    plt.title(f'Landmarks 68', fontsize=fontsize, pad = pad)

    # Landmarks 29 and visibility visualization
    plt.subplot(4, 4, 15)
    plt.imshow(landmark_29_visibility_img)
    plt.axis('off')
    plt.title(f'Landmarks 29 and visibility', fontsize=fontsize, pad = pad)

    plt.subplot(4, 4, 16)
    pose_estimation_image = draw_axis(cropped_rgb_image, yaw_prediction.cpu().numpy(), pitch_prediction.cpu().numpy(), roll_prediction.cpu().numpy(), size=25)
    plt.imshow(pose_estimation_image)
    plt.axis('off')
    plt.title("Head Pose Estimation", fontsize=fontsize, pad = pad)

    plt.tight_layout()
    plt.savefig(
        os.path.join(results_dir, figure_name)
    )
    plt.close()


In [29]:
for image_index in range(len(input_tensors)):
    get_image_visualization(model, input_tensors[image_index], split_sizes, cropping_side_length = cropping_side_length, figure_name = f'image_visualization_{image_index}.png')

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.0..255.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.0..255.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.0..255.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.0..255.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.0..255.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.0..255.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.0..255.0].
Clipping input data to the valid range for imshow with RGB dat